In [1]:
import os
from uuid import uuid4

from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import ArrayType, MapType, StringType, StructField, StructType
from py4j.protocol import Py4JJavaError

# -----------------------------
# Spark / Glue / Iceberg config
# -----------------------------
conf = SparkConf()

conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
conf.set("spark.sql.iceberg.handle-timestamp-without-timezone", "true")
conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# Iceberg catalog "glue" -> GlueCatalog + S3FileIO
conf.set("spark.sql.catalog.glue", "org.apache.iceberg.spark.SparkCatalog")
conf.set("spark.sql.catalog.glue.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
conf.set("spark.sql.catalog.glue.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
conf.set("spark.sql.catalog.glue.warehouse", "s3://302709293635-bronze/lol/")

conf.set("spark.wap.branch", "audit_branch")
conf.set("spark.sql.iceberg.wap.branch", "audit_branch")

sc = SparkContext(conf=conf)
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
df_assembleias = spark.read.csv(r"/home/hadoop/workspace/jupyter_workspace/data/assembleias.csv", header=True)
df_assembleias.createOrReplaceTempView("consorcio_assembleias")
df_contemplacoes = spark.read.csv(r"/home/hadoop/workspace/jupyter_workspace/data/contemplacoes.csv", header=True)
df_contemplacoes.createOrReplaceTempView("consorcio_contemplacoes")
df_cotas = spark.read.csv(r"/home/hadoop/workspace/jupyter_workspace/data/cotas.csv", header=True)
df_cotas.createOrReplaceTempView("consorcio_cotas")
df_grupos = spark.read.csv(r"/home/hadoop/workspace/jupyter_workspace/data/grupos.csv", header=True)
df_grupos.createOrReplaceTempView("consorcio_grupos")
df_lances = spark.read.csv(r"/home/hadoop/workspace/jupyter_workspace/data/lances.csv", header=True)
df_lances.createOrReplaceTempView("consorcio_lances")
df_leads = spark.read.csv(r"/home/hadoop/workspace/jupyter_workspace/data/leads.csv", header=True)
df_leads.createOrReplaceTempView("consorcio_leads")
df_pagamentos = spark.read.csv(r"/home/hadoop/workspace/jupyter_workspace/data/pagamentos.csv", header=True)
df_pagamentos.createOrReplaceTempView("consorcio_pagamentos")
df_propostas = spark.read.csv(r"/home/hadoop/workspace/jupyter_workspace/data/propostas.csv", header=True)
df_propostas.createOrReplaceTempView("consorcio_propostas")

In [ ]:
spark.read.table("glue.silver.consorcio_assembleias").show()

In [13]:
spark.sql(
        f"""
        CREATE TABLE glue.silver.consorcio_propostas
        USING iceberg
        TBLPROPERTIES (
            'format-version' = '2',
            'write.format.default' = 'parquet',
            'write.wap.enabled'='true'
        )
        AS
        SELECT *
        FROM consorcio_propostas
        """
    )

DataFrame[]

In [ ]:
def env(name: str, default: str | None = None, required: bool = False) -> str:
    value = os.getenv(name, default)
    if required and not value:
        raise ValueError(f"Variavel de ambiente obrigatoria nao informada: {name}")
    return value


def build_paths() -> tuple[str, str]:
    bronze_match_path = os.getenv("BRONZE_MATCH_PATH")
    bronze_champions_path = os.getenv("BRONZE_CHAMPIONS_PATH")
    if bronze_match_path and bronze_champions_path:
        return bronze_match_path, bronze_champions_path

    s3_bucket = env("S3_BUCKET", required=True)
    s3_prefix = env("S3_PREFIX", "")
    base = f"s3://{s3_bucket}/{s3_prefix}".rstrip("/")
    return f"{base}/match_details/", f"{base}/champions_details/"


def table_exists(spark: SparkSession, namespace: str, table_name: str) -> bool:
    # namespace deve ser algo como: glue.silver
    return spark.sql(f"SHOW TABLES IN {namespace} LIKE '{table_name}'").count() > 0


def ensure_match_table(spark: SparkSession, target_table: str) -> None:
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {target_table} (
            match_id STRING,
            participant_id STRING,
            riot_id_game_name STRING,
            riot_id_game_tag STRING,
            game_start_datetime TIMESTAMP,
            game_duration BIGINT,
            game_end_datetime TIMESTAMP,
            team_id BIGINT,
            game_result_first_team BOOLEAN,
            game_result_second_team BOOLEAN,
            champion_id BIGINT,
            champ_level BIGINT,
            individual_position STRING,
            deaths BIGINT,
            assists BIGINT,
            kills BIGINT,
            role STRING,
            kda DOUBLE,
            damage_per_minute DOUBLE,
            gold_per_minute DOUBLE,
            kill_participation DOUBLE,
            ability_uses BIGINT,
            baron_takedowns BIGINT,
            dodge_skill_shots_small_window BIGINT,
            dodge_skill_shots BIGINT,
            skill_shots_hit BIGINT,
            solo_kills BIGINT,
            spell1_casts BIGINT,
            spell2_casts BIGINT,
            spell3_casts BIGINT,
            spell4_casts BIGINT,
            item0 BIGINT,
            item1 BIGINT,
            item2 BIGINT,
            item3 BIGINT,
            item4 BIGINT,
            item5 BIGINT,
            item6 BIGINT,
            win BOOLEAN,
            processing_date DATE,
            updated_at TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (months(game_start_datetime))
        TBLPROPERTIES ('write.wap.enabled'='true')
        """
    )


def ensure_champions_table(spark: SparkSession, target_table: str) -> None:
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {target_table} (
            champion_id STRING,
            champion_key STRING,
            champion_name STRING,
            champion_title STRING,
            champion_tags ARRAY<STRING>,
            champion_partype STRING,
            champion_version STRING,
            source_version STRING,
            processing_date DATE,
            raw_payload STRING,
            updated_at TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (months(processing_date))
        TBLPROPERTIES ('write.wap.enabled'='true')
        """
    )


def merge_df(spark: SparkSession, df, target_table: str, keys: list[str]) -> None:
    """
    MERGE robusto:
    - Usa UPDATE SET * / INSERT * para evitar problemas com colunas reservadas (ex: role)
    - Coloca backticks nas chaves no ON
    - Imprime a exceção Java real se falhar (para debug)
    """
    temp_view = f"tmp_{uuid4().hex}"
    df.createOrReplaceTempView(temp_view)

    on_clause = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])

    sql = f"""
        MERGE INTO {target_table} t
        USING {temp_view} s
        ON {on_clause}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """

    try:
        spark.sql(sql)
    except Py4JJavaError as e:
        # Mostra o erro real vindo do Spark/Iceberg
        print("MERGE SQL:\n", sql)
        print("\nJAVA EXCEPTION:\n", e.java_exception)
        raise
    finally:
        spark.catalog.dropTempView(temp_view)


# -----------------------------
# DataFrames
# -----------------------------
def build_match_df(spark: SparkSession, bronze_match_path: str):
    raw = spark.read.option("multiLine", "true").json(bronze_match_path)

    array_cols = [name for name, dtype in raw.dtypes if dtype.startswith("array<")]
    if not array_cols:
        raise ValueError(
            "Nao foi encontrada a estrutura esperada no JSON de match_details (arrays na raiz)."
        )

    matches = (
        raw.select(F.explode(F.array(*[F.col(c) for c in array_cols])).alias("matches_array"))
        .select(F.explode("matches_array").alias("match"))
    )

    base = (
        matches.withColumn("participant", F.explode(F.col("match.info.participants")))
        .withColumn(
            "game_start_ms",
            F.coalesce(F.col("match.info.gameStartTimestamp"), F.col("match.info.gameCreation")),
        )
    )

    return base.select(
        F.col("match.metadata.matchId").alias("match_id"),
        F.col("participant.puuid").alias("participant_id"),
        F.col("participant.riotIdGameName").alias("riot_id_game_name"),
        F.col("participant.riotIdTagline").alias("riot_id_game_tag"),
        F.to_timestamp((F.col("game_start_ms") / F.lit(1000)).cast("double")).alias(
            "game_start_datetime"
        ),
        F.col("match.info.gameDuration").cast("bigint").alias("game_duration"),
        F.to_timestamp((F.col("match.info.gameEndTimestamp") / F.lit(1000)).cast("double")).alias(
            "game_end_datetime"
        ),
        F.col("participant.teamId").cast("bigint").alias("team_id"),
        F.col("match.info.teams")[0]["win"].alias("game_result_first_team"),
        F.col("match.info.teams")[1]["win"].alias("game_result_second_team"),
        F.col("participant.championId").cast("bigint").alias("champion_id"),
        F.col("participant.champLevel").cast("bigint").alias("champ_level"),
        F.col("participant.individualPosition").alias("individual_position"),
        F.col("participant.deaths").cast("bigint").alias("deaths"),
        F.col("participant.assists").cast("bigint").alias("assists"),
        F.col("participant.kills").cast("bigint").alias("kills"),
        F.col("participant.role").alias("role"),
        F.col("participant.challenges.kda").cast("double").alias("kda"),
        F.col("participant.challenges.damagePerMinute").cast("double").alias("damage_per_minute"),
        F.col("participant.challenges.goldPerMinute").cast("double").alias("gold_per_minute"),
        F.col("participant.challenges.killParticipation").cast("double").alias(
            "kill_participation"
        ),
        F.col("participant.challenges.abilityUses").cast("bigint").alias("ability_uses"),
        F.col("participant.challenges.baronTakedowns").cast("bigint").alias("baron_takedowns"),
        F.col("participant.challenges.dodgeSkillShotsSmallWindow").cast("bigint").alias(
            "dodge_skill_shots_small_window"
        ),
        F.col("participant.challenges.skillshotsDodged").cast("bigint").alias(
            "dodge_skill_shots"
        ),
        F.col("participant.challenges.skillshotsHit").cast("bigint").alias("skill_shots_hit"),
        F.col("participant.challenges.soloKills").cast("bigint").alias("solo_kills"),
        F.col("participant.spell1Casts").cast("bigint").alias("spell1_casts"),
        F.col("participant.spell2Casts").cast("bigint").alias("spell2_casts"),
        F.col("participant.spell3Casts").cast("bigint").alias("spell3_casts"),
        F.col("participant.spell4Casts").cast("bigint").alias("spell4_casts"),
        F.col("participant.item0").cast("bigint").alias("item0"),
        F.col("participant.item1").cast("bigint").alias("item1"),
        F.col("participant.item2").cast("bigint").alias("item2"),
        F.col("participant.item3").cast("bigint").alias("item3"),
        F.col("participant.item4").cast("bigint").alias("item4"),
        F.col("participant.item5").cast("bigint").alias("item5"),
        F.col("participant.item6").cast("bigint").alias("item6"),
        F.col("participant.win").cast("boolean").alias("win"),
        F.current_date().alias("processing_date"),
        F.current_timestamp().alias("updated_at"),
    )


def build_champions_df(spark: SparkSession, bronze_champions_path: str):
    raw = spark.read.option("multiLine", "true").json(bronze_champions_path)

    champion_value_schema = StructType(
        [
            StructField("id", StringType(), True),
            StructField("key", StringType(), True),
            StructField("name", StringType(), True),
            StructField("title", StringType(), True),
            StructField("tags", ArrayType(StringType()), True),
            StructField("partype", StringType(), True),
            StructField("version", StringType(), True),
        ]
    )
    champion_map_schema = MapType(StringType(), champion_value_schema, True)

    data_as_map = F.from_json(F.to_json(F.col("data")), champion_map_schema)
    exploded = raw.select(
        F.col("version").alias("source_version"),
        F.explode(data_as_map).alias("champion_name_key", "champion"),
    )

    return exploded.select(
        F.col("champion.id").alias("champion_id"),
        F.col("champion.key").alias("champion_key"),
        F.coalesce(F.col("champion.name"), F.col("champion_name_key")).alias("champion_name"),
        F.col("champion.title").alias("champion_title"),
        F.col("champion.tags").alias("champion_tags"),
        F.col("champion.partype").alias("champion_partype"),
        F.col("champion.version").alias("champion_version"),
        F.col("source_version"),
        F.current_date().alias("processing_date"),
        F.to_json(F.col("champion")).alias("raw_payload"),
        F.current_timestamp().alias("updated_at"),
    )


# -----------------------------
# Main
# -----------------------------
def main(spark: SparkSession):
    catalog = env("ICEBERG_CATALOG", "glue")
    db_name = env("ICEBERG_SILVER_DB", "silver")
    match_table_name = env("ICEBERG_MATCH_TABLE", "match_details")
    champions_table_name = env("ICEBERG_CHAMPIONS_TABLE", "champions_details")

    # IMPORTANTE: usar o catalog + db
    namespace = f"{catalog}.{db_name}"
    match_table = f"{namespace}.{match_table_name}"
    champions_table = f"{namespace}.{champions_table_name}"

    bronze_match_path, bronze_champions_path = build_paths()

    print("Warehouse:", spark.conf.get(f"spark.sql.catalog.{catalog}.warehouse", None))
    print("Namespace:", namespace)
    print("Tables:", match_table, "|", champions_table)
    print("Paths:", bronze_match_path, "|", bronze_champions_path)

    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {namespace}")

    match_df = build_match_df(spark, bronze_match_path)
    champions_df = build_champions_df(spark, bronze_champions_path)

    if not table_exists(spark, namespace, match_table_name):
        ensure_match_table(spark, match_table)

    if not table_exists(spark, namespace, champions_table_name):
        ensure_champions_table(spark, champions_table)

    merge_df(
        spark=spark,
        df=match_df,
        target_table=match_table,
        keys=["match_id", "participant_id"],
    )

    merge_df(
        spark=spark,
        df=champions_df,
        target_table=champions_table,
        keys=["champion_id"],
    )

    print(
        {
            "status": "ok",
            "namespace": namespace,
            "match_table": match_table,
            "champions_table": champions_table,
            "bronze_match_path": bronze_match_path,
            "bronze_champions_path": bronze_champions_path,
        }
    )


if __name__ == "__main__":
    main(spark)